In [671]:
import pandas as pd
import unicodedata

In [672]:
class DataProfiler:
    """Classe générique pour diagnostiquer la qualité d'un DataFrame."""
    def __init__(self, df, nom: str = "Dataset"):
        if not isinstance(df, pd.DataFrame):
            raise TypeError("df doit être un pandas DataFrame")
        self.df = df.copy()
        self.nom = nom
        self._stats = {}

    @classmethod
    def depuis_csv(cls, chemin, nom=None, separateur=","):
        df = pd.read_csv(chemin, sep=separateur, skip_blank_lines=True)
        df = df.dropna(how='all')
        nom = nom or chemin.split("/")[-1].replace(".csv", "")
        return cls(df, nom)

    def __repr__(self):
        return (f"DataProfiler(nom={self.nom}\n"
                f"Nbr lignes = {self.df.shape[0]}\n"
                f"Nbr colonnes = {self.df.shape[1]})")

    def valeurs_manquantes(self):
        nb_valu_m = self.df.isna().sum()
        serie_manquants = nb_valu_m[nb_valu_m > 0]
        return pd.DataFrame(serie_manquants, columns=["Valeurs_manquantes"])

    def doublons(self):
        return int(self.df.duplicated().sum())

    def profiler(self):
        self._stats = {
            "nom": self.nom,
            "nb_lignes": self.df.shape[0],
            "nb_colonnes": self.df.shape[1],
            "nb_doublons": self.doublons(),
            "valeurs_manquantes": self.valeurs_manquantes(),
        }
        return self


    def valeurs_extremes(self, colonne: str = "Valeur"):
        
        if colonne not in self.df.columns:
            print(f"[{self.nom}] Attention : colonne '{colonne}' introuvable.")
            return pd.DataFrame()
        
        serie_numerique = pd.to_numeric(self.df[colonne], errors='coerce')
        nb_non_numeriques = serie_numerique.isna().sum() - self.df[colonne].isna().sum()
        if nb_non_numeriques > 0:
            print(f"[{self.nom}] Attention : {nb_non_numeriques} valeurs non numériques détectées dans '{colonne}' "
                f"(ex: '<0.1', 'N/A'...), exclues du calcul des extrêmes.")
        serie_valide = serie_numerique.dropna()
        if serie_valide.empty:
            print(f"[{self.nom}] Attention : aucune valeur numérique valide dans '{colonne}', "
                  f"calcul des extrêmes impossible.")
            return pd.DataFrame()
        negatives = self.df[serie_numerique < 0]
        q1 = serie_valide.quantile(0.25)
        q3 = serie_valide.quantile(0.75)
        iqr = q3 - q1
        seuil_haut = q3 + 1.5 * iqr
        valeurs_hautes = self.df[serie_numerique > seuil_haut]
        print(f"[{self.nom}] Valeurs négatives : {len(negatives)}")
        print(f"[{self.nom}] Valeurs atypiques élevées (> Q3 + 1.5*IQR = {seuil_haut:.2f}) : {len(valeurs_hautes)}")

        return pd.concat([negatives, valeurs_hautes]).drop_duplicates()

    def rapport(self):
        valeurs_manquantes = self._stats.get('valeurs_manquantes')
        nb_manquant_total = int(valeurs_manquantes["Valeurs_manquantes"].sum()) if valeurs_manquantes is not None else 0
        if valeurs_manquantes is None:
            detail = "non calculé (appeler profiler() avant rapport())"
        elif valeurs_manquantes.empty:
            detail = "Aucune valeur manquante"
        else:
            detail = str(valeurs_manquantes)
        print(
            f"nom_dataset      : {self._stats.get('nom')}\n"
            f"nb_lignes_total  : {self._stats.get('nb_lignes')}\n"
            f"nb_colonne_total : {self._stats.get('nb_colonnes')}\n"
            f"nb_doublons      : {self._stats.get('nb_doublons')}\n"
            f"nb_manquant      : {nb_manquant_total} valeur(s) manquante(s) au total\n"
            f"detail_manquants :\n{valeurs_manquantes}"
        )
        return self


class ProfileurFAO(DataProfiler):

    def __init__(self, df, nom: str = "Dataset", colonne_pays='Zone', colonne_unite='Unité'):
        super().__init__(df, nom)
        self.colonne_pays = colonne_pays
        self.colonne_unite = colonne_unite
        
    @staticmethod
    def _normaliser(texte: str) -> str:
        return unicodedata.normalize('NFC', texte)

    def _trouver_colonne(self, nom_colonne: str):
        colonnes_normalisees = {self._normaliser(c): c for c in self.df.columns}
        cible = self._normaliser(nom_colonne)
        return colonnes_normalisees.get(cible)

    def pays_couverts(self):
        col = self._trouver_colonne(self.colonne_pays)
        if col is None:
            print(f"[{self.nom}] Attention : colonne '{self.colonne_pays}' introuvable, "
                  f"couverture pays vide.")
            return set()
        return set(self._normaliser(str(v)) for v in self.df[col].dropna().unique())

    def comparer_couverture(self, other: "ProfileurFAO"):
        zones_self = self.pays_couverts()
        zones_other = other.pays_couverts()

        if not zones_self:
            print(f"[{self.nom}] Attention : aucune zone détectée, la comparaison n'est pas fiable.")
        if not zones_other:
            print(f"[{other.nom}] Attention : aucune zone détectée, la comparaison n'est pas fiable.")

        manquants_chez_other = zones_self - zones_other
        manquants_chez_self = zones_other - zones_self

        print(f"[{self.nom}] Présents ici mais absents de [{other.nom}] "
              f"({len(manquants_chez_other)}) : {manquants_chez_other}")
        print(f"[{self.nom}] Absents ici mais présents dans [{other.nom}] "
              f"({len(manquants_chez_self)}) : {manquants_chez_self}")

        return {
            "manquants_chez_other": manquants_chez_other,
            "manquants_chez_self": manquants_chez_self
        }

    def unites_utilisees(self):
        col = self._trouver_colonne(self.colonne_unite)
        if col is None:
            print(f"[{self.nom}] Attention : colonne '{self.colonne_unite}' introuvable.")
            return []
        return self.df[col].unique()
    
    def fiabilite_symbole(self, colonne_symbole: str = "Symbole"):
        col = self._trouver_colonne(colonne_symbole)
        if not col:
            print(f"[{self.nom}] Info : pas de colonne '{colonne_symbole}' dans ce fichier.")
            return pd.Series(dtype=float)
        repartition = self.df[col].value_counts(normalize=True) * 100
        print(f"[{self.nom}] Répartition des symboles (%) :\n{repartition.round(2)}")
        return repartition

    def diagnostic(self):
        self.profiler()
        self.rapport()
        print(f"[{self.nom}] Nombre de pays : {len(self.pays_couverts())}")
        print(f"[{self.nom}] Unités utilisées : {self.unites_utilisees()}")


In [673]:
fichiers= {
    "Vegetaux": "FAO/fr_vegetaux.csv",
    "Animaux": "FAO/fr_animaux.csv",
    "Cereales": "FAO/fr_cereales.csv",
    "Population": "FAO/fr_population.csv",
    "SousAlimentation": "FAO/fr_sousalimentation.csv"
}

profileurs = {}
for nom, chemin in fichiers.items():
    p = ProfileurFAO.depuis_csv(chemin, nom=nom)
    p.diagnostic()
    profileurs[nom] = p
    print("=" * 60)

nom_dataset      : Vegetaux
nb_lignes_total  : 104871
nb_colonne_total : 14
nb_doublons      : 0
nb_manquant      : 0 valeur(s) manquante(s) au total
detail_manquants :
Empty DataFrame
Columns: [Valeurs_manquantes]
Index: []
[Vegetaux] Nombre de pays : 175
[Vegetaux] Unités utilisées : ['Milliers de tonnes' 'kg' 'Kcal/personne/jour' 'g/personne/jour']
nom_dataset      : Animaux
nb_lignes_total  : 37166
nb_colonne_total : 14
nb_doublons      : 0
nb_manquant      : 0 valeur(s) manquante(s) au total
detail_manquants :
Empty DataFrame
Columns: [Valeurs_manquantes]
Index: []
[Animaux] Nombre de pays : 175
[Animaux] Unités utilisées : ['Milliers de tonnes' 'kg' 'Kcal/personne/jour' 'g/personne/jour']
nom_dataset      : Cereales
nb_lignes_total  : 891
nb_colonne_total : 14
nb_doublons      : 0
nb_manquant      : 0 valeur(s) manquante(s) au total
detail_manquants :
Empty DataFrame
Columns: [Valeurs_manquantes]
Index: []
[Cereales] Nombre de pays : 167
[Cereales] Unités utilisées : ['Milliers d

In [674]:
nom = "SousAlimentation"
chemin = "FAO/fr_sousalimentation.csv"

p = ProfileurFAO.depuis_csv(chemin, nom=nom)

p.diagnostic()
profileurs[nom] = p
print("=" * 60)

nom_dataset      : SousAlimentation
nb_lignes_total  : 1020
nb_colonne_total : 15
nb_doublons      : 0
nb_manquant      : 1435 valeur(s) manquante(s) au total
detail_manquants :
        Valeurs_manquantes
Valeur                 415
Note                  1020
[SousAlimentation] Nombre de pays : 204
[SousAlimentation] Unités utilisées : ['millions']


In [675]:
for nom, p in profileurs.items():
    print(f"\n=== {nom} ===")
    p.valeurs_extremes()


=== Vegetaux ===
[Vegetaux] Valeurs négatives : 712
[Vegetaux] Valeurs atypiques élevées (> Q3 + 1.5*IQR = 42.50) : 18360

=== Animaux ===
[Animaux] Valeurs négatives : 35
[Animaux] Valeurs atypiques élevées (> Q3 + 1.5*IQR = 27.50) : 6338

=== Cereales ===
[Cereales] Valeurs négatives : 0
[Cereales] Valeurs atypiques élevées (> Q3 + 1.5*IQR = 2049.50) : 128

=== Population ===
[Population] Valeurs négatives : 0
[Population] Valeurs atypiques élevées (> Q3 + 1.5*IQR = 68388.50) : 19

=== SousAlimentation ===
[SousAlimentation] Attention : 115 valeurs non numériques détectées dans 'Valeur' (ex: '<0.1', 'N/A'...), exclues du calcul des extrêmes.
[SousAlimentation] Valeurs négatives : 0
[SousAlimentation] Valeurs atypiques élevées (> Q3 + 1.5*IQR = 13.10) : 56


In [676]:
import itertools

noms = list(profileurs.keys())
for nom_a, nom_b in itertools.combinations(noms, 2):
    zones_a = profileurs[nom_a].pays_couverts()
    zones_b = profileurs[nom_b].pays_couverts()

    if not zones_a or not zones_b:
        print(f"[{nom_a} vs {nom_b}] Comparaison ignorée (couverture pays vide pour au moins un fichier)")
        print("-" * 60)
        continue

    profileurs[nom_a].comparer_couverture(profileurs[nom_b])
    print("-" * 60)

[Vegetaux] Présents ici mais absents de [Animaux] (0) : set()
[Vegetaux] Absents ici mais présents dans [Animaux] (0) : set()
------------------------------------------------------------
[Vegetaux] Présents ici mais absents de [Cereales] (8) : {'Bermudes', 'Chine', 'Chine - RAS de Macao', 'Samoa', 'Polynésie française', 'Kiribati', 'Saint-Kitts-et-Nevis', 'Islande'}
[Vegetaux] Absents ici mais présents dans [Cereales] (0) : set()
------------------------------------------------------------
[Vegetaux] Présents ici mais absents de [Population] (0) : set()
[Vegetaux] Absents ici mais présents dans [Population] (0) : set()
------------------------------------------------------------
[Vegetaux] Présents ici mais absents de [SousAlimentation] (0) : set()
[Vegetaux] Absents ici mais présents dans [SousAlimentation] (29) : {'Groenland', 'Palestine', 'Îles Cook', 'Libye', 'Soudan du Sud', 'Tonga', 'Îles Marshall', 'Guinée équatoriale', 'Andorre', 'Tuvalu', 'Nauru', 'Tokélaou', 'Samoa américaine

In [677]:
extremes_vegetaux = profileurs["Vegetaux"].valeurs_extremes()
extremes_vegetaux[['Zone','Produit','Élément','Valeur']].head(20)

[Vegetaux] Valeurs négatives : 712
[Vegetaux] Valeurs atypiques élevées (> Q3 + 1.5*IQR = 42.50) : 18360


,Zone,Produit,Élément,Valeur
2,Afghanistan,Blé,Variation de stock,-350.0
82,Afghanistan,Sucre Eq Brut,Variation de stock,-19.0
393,Afrique du Sud,Riz (Eq Blanchi),Variation de stock,-231.0
558,Afrique du Sud,Sucre Eq Brut,Variation de stock,-326.0
631,Afrique du Sud,Soja,Variation de stock,-40.0
676,Afrique du Sud,Graines de coton,Variation de stock,-20.0
1133,Albanie,Orge,Variation de stock,-2.0
1148,Albanie,Maïs,Variation de stock,-20.0
1272,Albanie,Haricots,Variation de stock,-2.0
1373,Albanie,Olives,Variation de stock,-10.0


In [678]:
for nom, p in profileurs.items():
    valeur_num = pd.to_numeric(p.df['Valeur'], errors='coerce')
    negatifs = p.df[valeur_num < 0]
    
    if negatifs.empty:
        print(f"[{nom}] Aucune valeur négative.")
        continue
    
    print(f"\n[{nom}] {len(negatifs)} valeur(s) négative(s) :")
    print(negatifs[['Zone', 'Produit', 'Élément', 'Valeur']].to_string())
    print(negatifs['Élément'].value_counts())


[Vegetaux] 712 valeur(s) négative(s) :
                                              Zone                   Produit                                                        Élément    Valeur
2                                      Afghanistan                       Blé                                             Variation de stock   -350.00
82                                     Afghanistan             Sucre Eq Brut                                             Variation de stock    -19.00
393                                 Afrique du Sud          Riz (Eq Blanchi)                                             Variation de stock   -231.00
558                                 Afrique du Sud             Sucre Eq Brut                                             Variation de stock   -326.00
631                                 Afrique du Sud                      Soja                                             Variation de stock    -40.00
676                                 Afrique du Sud          

In [679]:
p_veg = profileurs["Vegetaux"].df
valeur_num = pd.to_numeric(p_veg['Valeur'], errors='coerce')
negatifs_veg = p_veg[valeur_num < 0]
print(negatifs_veg['Élément'].value_counts())

Élément
Variation de stock                                               599
Disponibilité intérieure                                         105
Importations - Quantité                                            1
Exportations - Quantité                                            1
Nourriture                                                         1
Disponibilité alimentaire en quantité (kg/personne/an)             1
Disponibilité alimentaire (Kcal/personne/jour)                     1
Disponibilité de protéines en quantité (g/personne/jour)           1
Disponibilité de matière grasse en quantité (g/personne/jour)      1
Traitement                                                         1
Name: count, dtype: int64


In [680]:
petits = negatifs_veg[~negatifs_veg['Élément'].isin(['Variation de stock', 'Disponibilité intérieure'])]
print(petits[['Zone', 'Produit', 'Élément', 'Valeur']].to_string())

              Zone         Produit                                                        Élément  Valeur
51994        Japon          Avoine                                        Importations - Quantité -201.00
51996        Japon          Avoine                                        Exportations - Quantité  -41.00
52002        Japon          Avoine                                                     Nourriture -246.00
52003        Japon          Avoine         Disponibilité alimentaire en quantité (kg/personne/an)   -1.93
52004        Japon          Avoine                 Disponibilité alimentaire (Kcal/personne/jour)  -21.00
52005        Japon          Avoine       Disponibilité de protéines en quantité (g/personne/jour)   -0.37
52006        Japon          Avoine  Disponibilité de matière grasse en quantité (g/personne/jour)   -0.03
74769  Ouzbékistan  Fruits, Autres                                                     Traitement   -4.00


In [681]:
p_ani = profileurs["Animaux"].df
valeur_num = pd.to_numeric(p_ani['Valeur'], errors='coerce')
negatifs_ani = p_ani[valeur_num < 0]

print(f"[Animaux] {len(negatifs_ani)} valeur(s) négative(s) :")
print(negatifs_ani[['Zone', 'Produit', 'Élément', 'Valeur']].to_string())
print(negatifs_ani['Élément'].value_counts())

[Animaux] 35 valeur(s) négative(s) :
                Zone                 Produit                   Élément  Valeur
2175       Australie      Lait - Excl Beurre        Variation de stock  -106.0
2408        Autriche      Lait - Excl Beurre        Variation de stock   -55.0
2738         Bahamas     Viande de Volailles        Variation de stock    -1.0
3632        Belgique            Beurre, Ghee        Variation de stock    -8.0
3653        Belgique  Graisses Animales Crue        Variation de stock    -2.0
8251          Chypre                   Crème  Disponibilité intérieure    -1.0
8901      Costa Rica  Graisses Animales Crue        Variation de stock    -2.0
9735        Danemark        Viande de Suides        Variation de stock   -10.0
9779        Danemark            Beurre, Ghee        Variation de stock    -1.0
9846        Danemark      Lait - Excl Beurre        Variation de stock    -4.0
10048       Djibouti      Lait - Excl Beurre        Variation de stock   -15.0
12944       Fin

In [682]:
for nom, p in profileurs.items():
    print(f"[{nom}] exemples Année :", p.df['Année'].unique()[:5])

[Vegetaux] exemples Année : [2013]
[Animaux] exemples Année : [2013]
[Cereales] exemples Année : [2013]
[Population] exemples Année : [2013]
[SousAlimentation] exemples Année : ['2012-2014' '2013-2015' '2014-2016' '2015-2017' '2016-2018']


In [683]:
p_pop = profileurs["Population"]
col = p_pop._trouver_colonne("Symbole")
lignes_manquantes = p_pop.df[p_pop.df[col].isna()]
print(f"Nombre de lignes avec valeur manquante : {len(lignes_manquantes)}")
print(lignes_manquantes['Zone'].value_counts().head(10))  

Nombre de lignes avec valeur manquante : 174
Zone
Afghanistan           1
Afrique du Sud        1
Albanie               1
Algérie               1
Allemagne             1
Angola                1
Antigua-et-Barbuda    1
Arabie saoudite       1
Argentine             1
Arménie               1
Name: count, dtype: int64


In [684]:
p_pop.df.head()

,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
0,FBSH,Bilans Alimentaire (Ancienne méthodologie et p...,2,Afghanistan,511,Population totale,2501,Population,2013,2013,1000 personnes,30552,NaN,Donnée officielle
1,FBSH,Bilans Alimentaire (Ancienne méthodologie et p...,202,Afrique du Sud,511,Population totale,2501,Population,2013,2013,1000 personnes,52776,NaN,Donnée officielle
2,FBSH,Bilans Alimentaire (Ancienne méthodologie et p...,3,Albanie,511,Population totale,2501,Population,2013,2013,1000 personnes,3173,NaN,Donnée officielle
3,FBSH,Bilans Alimentaire (Ancienne méthodologie et p...,4,Algérie,511,Population totale,2501,Population,2013,2013,1000 personnes,39208,NaN,Donnée officielle
4,FBSH,Bilans Alimentaire (Ancienne méthodologie et p...,79,Allemagne,511,Population totale,2501,Population,2013,2013,1000 personnes,82727,NaN,Donnée officielle


In [685]:
p_pop.df[p_pop.df['Symbole'].notna()][['Zone', 'Symbole', 'Description du Symbole']]

,Zone,Symbole,Description du Symbole
33,Chine,A,"Agrégat, peut inclure des données officielles,..."


In [686]:
p_sa = profileurs["SousAlimentation"]
col = p_sa._trouver_colonne("Symbole")
print(p_sa.df[col].value_counts(dropna=False))

sa_ref = p_sa.df[p_sa.df['Année'] == '2012-2014'].copy()

print(f"Nombre de lignes pour 2012-2014 : {len(sa_ref)}")
print(f"Nombre de zones couvertes       : {sa_ref['Zone'].nunique()}")
print(sa_ref['Symbole'].value_counts(dropna=False))

valides = sa_ref[sa_ref['Symbole'] == 'F']
print(f"Lignes avec valeur exploitable (Symbole='F') : {len(valides)}")


Symbole
F     605
NR    230
NV    185
Name: count, dtype: int64
Nombre de lignes pour 2012-2014 : 204
Nombre de zones couvertes       : 204
Symbole
F     120
NR     47
NV     37
Name: count, dtype: int64
Lignes avec valeur exploitable (Symbole='F') : 120


In [687]:
p_veg = profileurs["Vegetaux"].df
p_cer = profileurs["Cereales"].df
produits_cereales = set(p_cer['Produit'].unique())
produits_vegetaux = set(p_veg['Produit'].unique())

print("produit Cereales non présents Vegetaux ：", len(produits_cereales - produits_vegetaux))
print("valeur commune ：", len(produits_cereales & produits_vegetaux))

produit Cereales non présents Vegetaux ： 0
valeur commune ： 9


In [688]:
print(profileurs["Vegetaux"].df['Année'].unique())
print(profileurs["Population"].df['Année'].unique())

[2013]
[2013]


In [689]:
zones_pop = profileurs["Population"].pays_couverts()
zones_sa  = profileurs["SousAlimentation"].pays_couverts()
print("Dans Population mais pas dans SousAlimentation :", zones_pop - zones_sa)
print("Dans SousAlimentation mais pas dans Population :", zones_sa - zones_pop)

Dans Population mais pas dans SousAlimentation : set()
Dans SousAlimentation mais pas dans Population : {'Groenland', 'Palestine', 'Îles Cook', 'Libye', 'Soudan du Sud', 'Tonga', 'Îles Marshall', 'Guinée équatoriale', 'Andorre', 'Tuvalu', 'Nauru', 'Tokélaou', 'Samoa américaines', 'Micronésie (États fédérés de)', 'Nioué', 'Singapour', 'Qatar', 'Palaos', 'République arabe syrienne', 'Bahreïn', 'Bhoutan', 'République démocratique du Congo', 'Comores', 'Porto Rico', 'Somalie', 'Papouasie-Nouvelle-Guinée', 'Érythrée', 'Burundi', 'Seychelles'}


In [690]:
for nom in ["Vegetaux", "Animaux", "Cereales"]:
    print(f"\n[{nom}]")
    profileurs[nom].fiabilite_symbole()


[Vegetaux]
[Vegetaux] Répartition des symboles (%) :
Symbole
S     64.06
Fc    35.37
A      0.57
Name: proportion, dtype: float64

[Animaux]
[Animaux] Répartition des symboles (%) :
Symbole
S     59.53
Fc    40.01
A      0.46
Name: proportion, dtype: float64

[Cereales]
[Cereales] Répartition des symboles (%) :
Symbole
S    100.0
Name: proportion, dtype: float64


In [691]:
# ============ merge ============


element_kcal = 'Disponibilité alimentaire (Kcal/personne/jour)'

# ---- 1. Vegetaux ----
df_veg = profileurs["Vegetaux"].df.copy()
df_veg = df_veg[df_veg['Élément'] == element_kcal]
df_veg = df_veg[~((df_veg['Zone'] == 'Japon') & (df_veg['Produit'] == 'Avoine'))]  # 排除已确认的脏数据
df_veg_agg = df_veg.groupby('Zone', as_index=False)['Valeur'].sum()
df_veg_agg = df_veg_agg.rename(columns={'Valeur': 'Kcal_vegetaux'})

# ---- 2. Animaux：----
df_ani = profileurs["Animaux"].df.copy()
df_ani = df_ani[df_ani['Élément'] == element_kcal]
df_ani_agg = df_ani.groupby('Zone', as_index=False)['Valeur'].sum()
df_ani_agg = df_ani_agg.rename(columns={'Valeur': 'Kcal_animaux'})

# ---- 3. Population：suppression Symbole ----
df_pop = profileurs["Population"].df.drop(columns=['Symbole']).copy()
df_pop = df_pop[['Zone', 'Valeur']].rename(columns={'Valeur': 'Population_milliers'})

# ---- 4. SousAlimentation：selection 2012-2014NR/NV ----
df_sa = profileurs["SousAlimentation"].df.copy()
df_sa = df_sa[df_sa['Année'] == '2012-2014']
df_sa = df_sa[df_sa['Symbole'] == 'F']
df_sa = df_sa[['Zone', 'Valeur']].rename(columns={'Valeur': 'SousAlimentation_millions'})
df_sa['SousAlimentation_millions'] = pd.to_numeric(
    df_sa['SousAlimentation_millions'], errors='coerce'
)
df_sa = df_sa.dropna(subset=['SousAlimentation_millions'])
# ---- 5. Jointure inner sur Zone ----
df_global = df_veg_agg.merge(df_ani_agg, on='Zone', how='inner')
df_global = df_global.merge(df_pop, on='Zone', how='inner')
df_global = df_global.merge(df_sa, on='Zone', how='inner')

# ---- 6. Indicateurs dérivés ----
df_global['Disponibilite_calorique_totale_kcal'] = (
    df_global['Kcal_vegetaux'] + df_global['Kcal_animaux']
)
df_global['Population_totale'] = df_global['Population_milliers'] * 1000
df_global['Taux_sousnutrition_pct'] = (
    (df_global['SousAlimentation_millions'] * 1_000_000) / df_global['Population_totale'] * 100
)

print(f"Dataset final : {len(df_global)} pays")
print(df_global.head())

# ---- 7. Documenter les pays exclus (obligatoire) ----
toutes_zones = (
    set(df_veg_agg['Zone']) | set(df_ani_agg['Zone']) |
    set(df_pop['Zone']) | set(df_sa['Zone'])
)
zones_finales = set(df_global['Zone'])
pays_exclus = sorted(toutes_zones - zones_finales)
print(f"\nPays exclus : {len(pays_exclus)}")
print(pays_exclus)

# ---- 8. Export CSV ----
df_global.to_csv('dataset_global_fao.csv', index=False)
print("\n✅ Export terminé : dataset_global_fao.csv")

Dataset final : 97 pays
             Zone  Kcal_vegetaux  Kcal_animaux  Population_milliers  \
0     Afghanistan         1871.0         216.0                30552   
1  Afrique du Sud         2533.0         487.0                52776   
2         Albanie         2203.0         985.0                 3173   
3         Algérie         2915.0         378.0                39208   
4          Angola         2221.0         253.0                21472   

   SousAlimentation_millions  Disponibilite_calorique_totale_kcal  \
0                        7.9                               2087.0   
1                        2.6                               3020.0   
2                        0.2                               3188.0   
3                        1.7                               3293.0   
4                        8.1                               2474.0   

   Population_totale  Taux_sousnutrition_pct  
0           30552000               25.857554  
1           52776000                4.92

In [692]:
for nom in ["Vegetaux", "Animaux", "Cereales"]:
    df = profileurs[nom].df
    dupes = df.duplicated(subset=['Zone', 'Produit', 'Élément']).sum()
    print(f"[{nom}] doublons sur clé métier (Zone,Produit,Élément) : {dupes}")

for nom in ["Population", "SousAlimentation"]:
    df = profileurs[nom].df
    dupes = df.duplicated(subset=['Zone', 'Année']).sum()
    print(f"[{nom}] doublons sur clé métier (Zone,Année) : {dupes}")

[Vegetaux] doublons sur clé métier (Zone,Produit,Élément) : 0
[Animaux] doublons sur clé métier (Zone,Produit,Élément) : 0
[Cereales] doublons sur clé métier (Zone,Produit,Élément) : 0
[Population] doublons sur clé métier (Zone,Année) : 0
[SousAlimentation] doublons sur clé métier (Zone,Année) : 0


In [693]:
# vérification rapide
sa_check = profileurs["SousAlimentation"].df
sa_2012_2014 = sa_check[sa_check['Année'] == '2012-2014']
print(f"nb ligne: {len(sa_2012_2014)}")
print(f"nb pays: {sa_2012_2014['Zone'].nunique()}")
print(sa_2012_2014['Symbole'].value_counts(dropna=False))

nb ligne: 204
nb pays: 204
Symbole
F     120
NR     47
NV     37
Name: count, dtype: int64


In [694]:
print(df_global.shape)

(97, 8)


In [695]:
print(df_global.head())

             Zone  Kcal_vegetaux  Kcal_animaux  Population_milliers  \
0     Afghanistan         1871.0         216.0                30552   
1  Afrique du Sud         2533.0         487.0                52776   
2         Albanie         2203.0         985.0                 3173   
3         Algérie         2915.0         378.0                39208   
4          Angola         2221.0         253.0                21472   

   SousAlimentation_millions  Disponibilite_calorique_totale_kcal  \
0                        7.9                               2087.0   
1                        2.6                               3020.0   
2                        0.2                               3188.0   
3                        1.7                               3293.0   
4                        8.1                               2474.0   

   Population_totale  Taux_sousnutrition_pct  
0           30552000               25.857554  
1           52776000                4.926482  
2            3173